In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12,6)

In [ ]:
df = pd.read_csv("/kaggle/input/quantum-circuit-fault-logs-100k-nisq-simulations/NISQ-FaultLogs-100K.csv")

In [ ]:
print("Dataset shape:", df.shape)
df.head()

In [ ]:
display(df.info())
display(df.describe().T)

## Validation Summary

**✅ Realistic quantum hardware behavior:**

- **Gate error rate**: ~0.1%–1.5% → realistic NISQ noise regime  
- **Readout error**: ~0.5%–3% → matches superconducting qubit ranges
- **T1/T2 coherence times**: mostly between 30–60 μs, consistent with real devices

**⚠️ Issue found:**

- Minimum values for T1 and T2 are **negative**, which is **physically impossible**

In [ ]:
num_cols = ["qubit_count","gate_depth","error_rate_gate","t1_time","t2_time","readout_error","fidelity"]

fig, axes = plt.subplots(len(num_cols),1, figsize=(12,22))

for i,col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

## 📊 Distribution Insights

**Qubit Count**
- Uniform from 5–15  
- Dataset balanced across circuit sizes

**Gate Depth**
- Right-skewed  
- Most circuits shallow (<30 gates)  
- Few deep circuits → rare cases

**Gate Error Rate**
- Mostly between 0.003–0.007  
- Realistic noise region

**T1 / T2 Times**
- Roughly normal distribution  
- Centered near realistic hardware ranges

**Readout Error**
- Slight right skew  
- Majority low-noise measurements

**Fidelity**
- Highly skewed toward low values  
- Most circuits noisy → realistic NISQ behavior

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

## 🔥 Correlation Insights

**Strong Relationships**
- T1 vs T2 → **0.89** → Physically expected (coherence linked)
- Bitstring vs Ideal → **0.55** → Noisy outputs still partially follow ideal

**Most Important Finding**
- Qubit Count vs Fidelity → **-0.93**
→ Larger circuits drastically reduce fidelity

**Weak / Near Zero**
- Gate depth surprisingly uncorrelated with fidelity
- Error rate not strongly linearly correlated

**Interpretation**
→ Noise impact is dominated by system size, not just gate errors.

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df, x="qubit_count", y="fidelity")
plt.title("Fidelity vs Qubit Count")
plt.show()


## 📉 Fidelity vs Qubit Count

- Clear monotonic drop in fidelity as qubits increase
- Small circuits (5–7 qubits) maintain high fidelity
- Large circuits (>12 qubits) almost always low fidelity

**Key Insight**
→ Noise scales strongly with system size.

**Competition Observation**
Scaling qubit count is the dominant failure factor in NISQ devices.

In [ ]:
plt.figure(figsize=(12,7))
sns.scatterplot(
    data=df.sample(15000),
    x="gate_depth",
    y="fidelity",
    hue="qubit_count",
    palette="viridis",
    alpha=0.6
)
plt.title("Fidelity vs Gate Depth colored by Qubit Count")
plt.show()

## 🎯 Fidelity vs Gate Depth (Colored by Qubit Count)

- Gate depth alone does not strongly determine fidelity
- Color gradient shows larger qubit systems cluster near zero fidelity
- Small-qubit circuits maintain fidelity even at higher depths

**Key Insight**
→ System size matters more than circuit depth.

**Hidden Pattern**
Large circuits fail regardless of depth, while small circuits remain robust.

In [ ]:
plt.figure(figsize=(12,6))
sns.violinplot(data=df, x="device_type", y="fidelity")
plt.title("Fidelity Distribution by Device Type")
plt.show()

## 🧪 Fidelity by Device Type

- Both hardware types show very similar distributions
- Majority of runs cluster near low fidelity
- High-fidelity outcomes are rare for both

**Key Insight**
→ Noise impact is architecture-agnostic in this dataset.

**Takeaway**
Device type is not a strong standalone predictor of fidelity.

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df.sample(15000),
    x="error_rate_gate",
    y="fidelity",
    alpha=0.5
)
plt.title("Fidelity vs Gate Error Rate")
plt.show()

## ⚙️ Fidelity vs Gate Error Rate

- No strong linear trend visible
- High and low fidelity occur across all error rates
- Distribution looks vertically spread

**Key Insight**
→ Gate error rate alone does not determine fidelity.

**Hidden Pattern**
Fidelity depends on multiple interacting factors, not a single noise parameter.

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df.sample(15000),
    x="t1_time",
    y="fidelity",
    alpha=0.5
)
plt.title("Fidelity vs T1 Coherence Time")
plt.show()

## ⏱️ Fidelity vs T1 Coherence

- No clear upward trend
- High coherence does not guarantee high fidelity
- Low fidelity appears across all T1 ranges

**Key Insight**
→ Longer coherence alone is not sufficient for good circuit performance.

**Interpretation**
Circuit errors are dominated by system complexity + noise interactions, not just relaxation time.

In [ ]:
df["complexity"] = df["qubit_count"] * df["gate_depth"]

plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df.sample(15000),
    x="complexity",
    y="fidelity",
    alpha=0.5
)
plt.title("Fidelity vs Circuit Complexity (Qubits × Depth)")
plt.show()

## 🚀 Fidelity vs Circuit Complexity

- Clear downward trend as complexity increases
- High-complexity circuits almost always near zero fidelity
- Low-complexity circuits show wide performance range

**Key Insight**
→ Fidelity collapses as circuit complexity grows.

**Major Finding**
System complexity (qubits × depth) is the dominant predictor of failure.

In [ ]:
df["log_complexity"] = np.log1p(df["complexity"])

plt.figure(figsize=(12,6))
sns.regplot(
    data=df.sample(15000),
    x="log_complexity",
    y="fidelity",
    scatter_kws={"alpha":0.4}
)
plt.title("Fidelity vs Log Circuit Complexity")
plt.show()

## 📉 Fidelity vs Log Complexity

- Clear linear downward trend
- Relationship becomes smoother after log transform
- Variance decreases as complexity increases

**Key Insight**
→ Log complexity is strongly predictive of fidelity.

**Modeling Insight**
Log-transformed interaction features capture system physics better than raw variables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features = [
    "qubit_count",
    "gate_depth",
    "error_rate_gate",
    "t1_time",
    "t2_time",
    "readout_error",
    "complexity",
    "log_complexity"
]

X = df[features]
y = df["fidelity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=150, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

## 🤖 Baseline Model Performance

Random Forest Results:

- MAE: 0.045
- R²: 0.937

**Interpretation**
Model explains ~94% of fidelity variance.

**Key Insight**
Fidelity is highly predictable from circuit + noise parameters.

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Feature Importance")
plt.show()

## 🧠 Feature Importance Analysis

- Qubit count dominates all other features
- Every other variable contributes minimally

**Key Insight**
→ System size is the primary driver of fidelity loss.

**Scientific Interpretation**
Scaling quantum systems introduces exponentially harder error control.

**Modeling Insight**
Most predictive power comes from a single variable → problem is structurally simple.

## Final Conclusions

This dataset reveals a clear physical law:

> Circuit size determines success.

Major findings:

- Fidelity sharply decreases as qubit count increases
- Circuit complexity strongly predicts failure
- Noise parameters alone cannot explain performance
- Log-transformed interaction features improve modeling
- A simple Random Forest achieves R² ≈ 0.94

**Final Insight**
NISQ-era performance is fundamentally limited by scale, not just noise.

---

## 🚀 Future Work

Possible directions:

- anomaly detection for hardware faults
- drift detection over timestamps
- generative modeling of realistic circuits
- quantum-aware ML architectures

---

## 💬 Community Note

If you explored this dataset, I’d love to hear your insights!

- What patterns did you discover?
- Did you find any unexpected relationships?
- Would you model it differently?

If you found this notebook helpful or interesting, consider **upvoting the notebook and dataset** — it really helps visibility and encourages more deep-dive analyses like this 🙌